# Munge summary statistics 
1. Prep munging script and gzip GP2 data 
2. Use gwaslab to prep data
3. Prep + munge MVP data
4. Prep + munge 23andMe data

In [8]:
## using python 3.10 custom gwaslab env 
## Import the necessary packages 
import os
import numpy as np
import pandas as pd
import gwaslab as gl
import math
import sys
import subprocess
import statsmodels.api as sm
import scipy
from scipy import stats
from scipy.stats import chi2
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

## Print out package versions
## Getting packages loaded into this notebook and their versions to allow for reproducibility
import pkg_resources
import types
from datetime import date

today = date.today()
date = today.strftime("%d-%b-%Y").upper()

## Define function 
def get_imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            name = val.__name__.split(".")[0]
        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        poorly_named_packages = {
            "PIL": "Pillow",
            "sklearn": "scikit-learn"
        }
        if name in poorly_named_packages:
            name = poorly_named_packages[name]

        yield name

## Get a list of packages imported 
imports = list(set(get_imports()))

requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name != "pip":
        requirements.append((m.project_name, m.version))

## Print out packages and versions 
print(f"PACKAGE VERSIONS ({date})")
for r in requirements:
    print("\t{}=={}".format(*r))

## Also print which Python is being used
print("\nPYTHON INFO")
print(f"\tPython executable: {sys.executable}")

PACKAGE VERSIONS (18-NOV-2025)
	seaborn==0.13.2
	statsmodels==0.14.4
	gwaslab==3.6.8
	matplotlib==3.8.4
	numpy==1.26.4
	pandas==2.3.2
	scipy==1.15.3

PYTHON INFO
	Python executable: /vf/users/makariousmb/conda/envs/gwaslab_310/bin/python


In [ ]:
# setting references 
gl.options.set_option("config","/data/gwaslab/data/config.json")
gl.options.set_option("reference","/data/gwaslab/data/reference.json")
gl.options.set_option("formatbook","/data/gwaslab/data/formatbook.json")
gl.options.set_option("data_directory","/data/gwaslab/.gwaslab/")

In [ ]:
# available_ref = gl.check_available_ref()

# gl.download_ref("ucsc_genome_hg38")
# gl.download_ref("1kg_eur_hg38")
# gl.download_ref("1kg_afr_hg38")
# gl.download_ref("1kg_dbsnp151_hg38_auto")

In [ ]:
print(gl.get_path("ucsc_genome_hg38"))
print(gl.get_path("1kg_afr_hg38"))

# Munging script (with allele frequencies)
Modified from Hampton L.

In [ ]:
%%writefile ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_test_alleles.py
#!/usr/bin/env python3

import pandas as pd
import numpy as np
from tqdm import tqdm
import gwaslab as gl
import logging
import sys
import time
import os
import argparse

'''
/data/conda/envs/gwaslab_310/bin/python ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_test_alleles.py \
  --input /path/to/gzipped/sumstats.txt.gz \
  --output /path/to/completed/munged/stats.wAlleles.FOR_PLINK.txt \
  --log /path/to/save/log.wAlleles.FOR_PLINK.log
'''

def setup_logging(logfile):
    """Setup logging that flushes immediately to file + console"""
    os.makedirs(os.path.dirname(logfile), exist_ok=True)
    open(logfile, "a").close()  # create file immediately

    # File handler
    file_handler = logging.FileHandler(logfile, mode="a")
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

    root = logging.getLogger()
    root.setLevel(logging.INFO)
    root.handlers = []  # clear old handlers
    root.addHandler(file_handler)
    root.addHandler(console_handler)


def log_rows(step, before, after, elapsed=None):
    removed = before - after
    msg = f"{step}: {before:,} → {after:,} (removed {removed:,})"
    if elapsed is not None:
        msg += f" | time: {elapsed:.2f}s"
    logging.info(msg)


def log_step(step, elapsed=None):
    msg = step
    if elapsed is not None:
        msg += f" | time: {elapsed:.2f}s"
    logging.info(msg)


def main():
    parser = argparse.ArgumentParser(description="Munge and harmonize GWAS summary stats for PLINK input.")
    parser.add_argument("--input", required=True, help="Path to input gzipped summary stats file")
    parser.add_argument("--output", required=True, help="Path to output harmonized file (tsv)")
    parser.add_argument("--log", required=True, help="Path to log file")
    args = parser.parse_args()

    setup_logging(args.log)

    global_start = time.perf_counter()

    # === Load data ===
    t0 = time.perf_counter()
    logging.info(f"Reading compressed input file: {args.input}")
    munge = pd.read_csv(args.input, sep='\t', compression='gzip')
    log_step(f"Initial rows: {len(munge):,}", time.perf_counter() - t0)

    # === Filter ERRCODE ===
    t0 = time.perf_counter()
    before = len(munge)
    munge = munge[munge['ERRCODE'] == '.']
    log_rows("After removing ERRCODE != .", before, len(munge), time.perf_counter() - t0)

    # === Determine alternate allele ===
    t0 = time.perf_counter()
    munge['A2'] = np.where(munge['A1'] == munge['ALT'], munge['REF'], munge['ALT'])
    log_step("Computed A2", time.perf_counter() - t0)

    # === Clean up OR values ===
    munge['OR'] = pd.to_numeric(munge['OR'], errors="coerce")

    for step, condition in [
        # ("After dropping OR NaN", munge['OR'].notna()),
        ("After dropping OR == 1", munge['OR'] != 1),
        ("After dropping OR == 0", munge['OR'] != 0),
    ]:
        t0 = time.perf_counter()
        before = len(munge)
        munge = munge[condition]
        log_rows(step, before, len(munge), time.perf_counter() - t0)

    # === Compute BETA ===
    t0 = time.perf_counter()
    munge['BETA'] = np.log(munge['OR'])
    log_step("Computed BETA = log(OR)", time.perf_counter() - t0)

    # === Compute N and NMISS ===
    t0 = time.perf_counter()
    munge['CASE_ALLELE_CT'] = munge['CASE_ALLELE_CT'].astype(float)
    munge['CTRL_ALLELE_CT'] = munge['CTRL_ALLELE_CT'].astype(float)
    munge['ncases'] = munge['CASE_ALLELE_CT'] / 2
    munge['ncontrols'] = munge['CTRL_ALLELE_CT'] / 2
    munge['NMISS'] = 4 / (1 / (munge['ncases'].astype(int)) + 1 / (munge['ncontrols'].astype(int)))
    munge['OBS_CT'] = munge['ncases'].astype(int) + munge['ncontrols'].astype(int)
    log_step("Computed counts and NMISS", time.perf_counter() - t0)

    # === Frequency + region filtering ===
    t0 = time.perf_counter()
    before = len(munge)
    condition1 = (munge['A1_FREQ'].astype(float).between(0.01, 0.99))
    condition2 = (
        ((munge['#CHROM'].astype(str) == '4') & (munge['POS'].astype(float).between(89674099, 89888304))) |
        ((munge['#CHROM'].astype(str) == '1') & (munge['POS'].astype(float).between(155184452, 155294627))) |
        ((munge['#CHROM'].astype(str) == '12') & (munge['POS'].astype(float).between(40174997, 40419285)))
    )
    munge = munge[condition1 | condition2]
    log_rows("After frequency + region filtering", before, len(munge), time.perf_counter() - t0)

    # === Harmonization ===
    t0 = time.perf_counter()
    logging.info("Initializing harmonization with gwaslab...")
    mysumstats = gl.Sumstats(
        munge,
        snpid="ID",
        chrom="#CHROM",
        pos="POS",
        p="P",
        build="38",
        nea="A2",
        ea="A1",
        eaf="A1_FREQ",
        beta="BETA",
        se="LOG(OR)_SE",
        n="OBS_CT",
        other=["NMISS"]
    )

    logging.info("Running harmonization (can take a while)...")
    mysumstats.harmonize(
        basic_check=True,
        n_cores=1,
        ref_seq="/data/makariousmb/gwaslab/.gwaslab/hg38.fa",
        ref_infer="/data/makariousmb/gwaslab/.gwaslab/AFR.ALL.split_norm_af.1kg_30x.hg38.vcf.gz",
        ref_alt_freq="AF"
    )
    log_step("Harmonization complete", time.perf_counter() - t0)

    # === Extract harmonized data ===
    t0 = time.perf_counter()
    sumdf = mysumstats.data
    log_step(f"Extracted harmonized dataframe with {len(sumdf):,} rows", time.perf_counter() - t0)

    # === NEW_ID generation + dedup ===
    t0 = time.perf_counter()
    logging.info("Generating NEW_IDs")
    tqdm.pandas(desc="Creating NEW_IDs")
    sumdf['NEW_ID'] = (
        "chr" + sumdf['CHR'].astype(str).progress_map(str)
        + ":" + sumdf['POS'].astype(str)
        + ":" + sumdf['NEA'].astype(str)
        + ":" + sumdf['EA'].astype(str)
    )
    log_step("NEW_ID generation done", time.perf_counter() - t0)

    t0 = time.perf_counter()
    before = len(sumdf)
    sumdf_nodup = sumdf.drop_duplicates(subset=['NEW_ID'])
    log_rows("After dropping duplicate NEW_IDs", before, len(sumdf_nodup), time.perf_counter() - t0)

    # === Final output (with EAF) ===
    t0 = time.perf_counter()
    sumdf_nodup = sumdf_nodup[['NEW_ID', 'CHR', 'POS', 'EA', 'NEA', 'EAF', 'BETA', 'SE', 'P', 'NMISS']]
    sumdf_nodup.columns = ['SNP', 'CHR', 'BP', 'A1', 'A2', 'EAF', 'BETA', 'SE', 'P', 'NMISS']

    logging.info(f"Saving output to {args.output}...")
    sumdf_nodup.to_csv(args.output, index=False, sep="\t", header=True)
    log_step(f"Done | total rows: {len(sumdf_nodup):,}", time.perf_counter() - t0)

    # === Total runtime ===
    total_time = time.perf_counter() - global_start
    logging.info(f"=== Pipeline finished in {total_time/60:.2f} minutes ({total_time:.2f} seconds) ===")


if __name__ == "__main__":
    main()

# GP2 Data

## Prep swarm files

In [ ]:
%%bash

cat << EOF > ${WORK_DIR}/scripts/munge_sum_stats_AAC.swarm
/data/conda/envs/gwaslab_310/bin/python /data/CARD_AA/projects/2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_test_alleles.py \\
  --input ${WORK_DIR}/results/GP2_R11/AAC/AAC_release11_vwb.PHENO.hybrid.ALLCHR.filtered.txt.gz \\
  --output ${WORK_DIR}/data/GP2_R11/AAC/GP2_AAC_GWAS_R11.wAlleles.FOR_PLINK.txt \\
  --log ${WORK_DIR}/scripts/GP2_AAC_GWAS_R11.wAlleles.FOR_PLINK.log
EOF

In [ ]:
%%bash

cat << EOF > ${WORK_DIR}/scripts/munge_sum_stats_AFR.swarm
/data/conda/envs/gwaslab_310/bin/python /data/CARD_AA/projects/2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_test_alleles.py \\
  --input ${WORK_DIR}/results/GP2_R11/AFR/AFR_release11_vwb.PHENO.hybrid.ALLCHR.filtered.txt.gz \\
  --output ${WORK_DIR}/data/GP2_R11/AFR/GP2_AFR_GWAS_R11.wAlleles.FOR_PLINK.txt \\
  --log ${WORK_DIR}/scripts/GP2_AFR_GWAS_R11.wAlleles.FOR_PLINK.log
EOF

## Submit swarm files

In [ ]:
%%bash

cd ${WORK_DIR}/scripts/

swarm --file ${WORK_DIR}/scripts/munge_sum_stats_AAC.swarm \
-g 100 --time 6:00:00 \
--sbatch "--mail-type=BEGIN,FAIL,TIME_LIMIT_80,END"

4746785


In [ ]:
%%bash

cd ${WORK_DIR}/scripts/

swarm --file ${WORK_DIR}/scripts/munge_sum_stats_AFR.swarm \
-g 100 --time 6:00:00 \
--sbatch "--mail-type=BEGIN,FAIL,TIME_LIMIT_80,END"

4746789


In [28]:
! head {WORK_DIR}/data/GP2_R11/AFR/GP2_AFR_GWAS_R11.wAlleles.FOR_PLINK.txt

SNP	CHR	BP	A1	A2	EAF	BETA	SE	P	NMISS
chr1:730869:C:T	1	730869	T	C	0.0137411	-0.027717608279898675	0.229203	0.903745	6273.823933154283
chr1:758443:G:C	1	758443	C	G	0.243619	-0.1789795475332699	0.0536287	0.000845743	6273.823933154283
chr1:763097:C:T	1	763097	T	C	0.0690303	-0.014614270129058726	0.0837582	0.861491	6273.823933154283
chr1:763668:T:C	1	763668	C	T	0.0480593	-0.06970276625977379	0.106527	0.512905	6273.823933154283
chr1:764648:A:G	1	764648	G	A	0.0342894	-0.045458769432091234	0.130514	0.72761	6273.823933154283
chr1:767578:T:C	1	767578	C	T	0.0880746	0.09021540527130362	0.0758576	0.234312	6273.823933154283
chr1:769809:A:C	1	769809	C	A	0.0147032	-0.28482881353586925	0.194659	0.143406	6273.823933154283
chr1:770072:T:A	1	770072	A	T	0.212383	0.07789578930808588	0.0554385	0.160016	6273.823933154283
chr1:770594:C:T	1	770594	T	C	0.0167497	-0.022639346983665168	0.187445	0.903866	6273.823933154283


# 23andMe

In [ ]:
## Using unfiltered summary stats 
! head {DATA_DIR}/preprocessing/filtered_sumstats_23andme_AFRICAN_PD.txt

In [ ]:
%%writefile ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_23andMe.py
#!/usr/bin/env python3.10

import pandas as pd
import numpy as np
from tqdm import tqdm
import gwaslab as gl
import logging
import sys
import time
import os
import argparse

"""
Example:

/data/conda/envs/gwaslab_310/bin/python \
  ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_23andMe.py \
  --input /path/to/23andme_sumstats.txt[.gz] \
  --output /path/to/munged/stats.wAlleles.FOR_PLINK.txt \
  --log /path/to/logs/stats.wAlleles.FOR_PLINK.log

NOTE:
- This version expects 23andMe-style columns, e.g.:
    assay.name, scaffold, position, effect_allele, alt_allele,
    effect, stderr, pvalue, pass, n_cases, n_controls, freq.a, ...
- Make sure gwaslab references are available:
    gl.download_ref("ucsc_genome_hg38")
    gl.download_ref("1kg_afr_hg38")
"""

gl.options.set_option("config","/data/gwaslab/data/config.json")
gl.options.set_option("reference","/data/gwaslab/data/reference.json")
gl.options.set_option("formatbook","/data/gwaslab/data/formatbook.json")
gl.options.set_option("data_directory","/data/gwaslab/.gwaslab/")

def setup_logging(logfile):
    """Setup logging that flushes immediately to file + console."""
    os.makedirs(os.path.dirname(logfile), exist_ok=True)
    open(logfile, "a").close()  # create file immediately

    file_handler = logging.FileHandler(logfile, mode="a")
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

    root = logging.getLogger()
    root.setLevel(logging.INFO)
    root.handlers = []  # clear old handlers
    root.addHandler(file_handler)
    root.addHandler(console_handler)


def log_rows(step, before, after, elapsed=None):
    removed = before - after
    msg = f"{step}: {before:,} → {after:,} (removed {removed:,})"
    if elapsed is not None:
        msg += f" | time: {elapsed:.2f}s"
    logging.info(msg)


def log_step(step, elapsed=None):
    msg = step
    if elapsed is not None:
        msg += f" | time: {elapsed:.2f}s"
    logging.info(msg)


def main():
    parser = argparse.ArgumentParser(
        description="Munge and harmonize 23andMe-style GWAS summary stats for PLINK input."
    )
    parser.add_argument("--input", required=True, help="Path to input summary stats file (tsv, can be gzipped)")
    parser.add_argument("--output", required=True, help="Path to output harmonized file (tsv)")
    parser.add_argument("--log", required=True, help="Path to log file")
    args = parser.parse_args()

    setup_logging(args.log)
    global_start = time.perf_counter()

    # === Load data ===
    t0 = time.perf_counter()
    logging.info(f"Reading input file (compression inferred): {args.input}")
    munge = pd.read_csv(args.input, sep="\t", compression="infer")
    log_step(f"Initial rows: {len(munge):,}", time.perf_counter() - t0)

    # === Filter: pass == 'Y' ===
    t0 = time.perf_counter()
    if "pass" not in munge.columns:
        logging.error(
            f"Expected a 'pass' column in the input, but did not find it.\n"
            f"Available columns: {list(munge.columns)}"
        )
        sys.exit(1)

    before = len(munge)
    munge = munge[munge["pass"] == "Y"]
    log_rows("After filtering pass == 'Y'", before, len(munge), time.perf_counter() - t0)

    # === Filter: effect != 0 ===
    t0 = time.perf_counter()
    before = len(munge)
    munge["effect"] = pd.to_numeric(munge["effect"], errors="coerce")
    munge = munge[munge["effect"] != 0]
    log_rows("After filtering effect != 0", before, len(munge), time.perf_counter() - t0)

    # === Compute NMISS and OBS_CT ===
    t0 = time.perf_counter()
    # Ensure numeric
    munge["n_cases"] = pd.to_numeric(munge["n_cases"], errors="coerce")
    munge["n_controls"] = pd.to_numeric(munge["n_controls"], errors="coerce")

    # Drop rows with NaN n_cases / n_controls before NMISS calc
    before = len(munge)
    munge = munge[munge["n_cases"].notna() & munge["n_controls"].notna()]
    log_rows("After dropping rows with missing n_cases/n_controls", before, len(munge))

    munge["NMISS"] = 4.0 / (1.0 / munge["n_cases"].astype(int) + 1.0 / munge["n_controls"].astype(int))
    munge["OBS_CT"] = munge["n_cases"].astype(int) + munge["n_controls"].astype(int)
    log_step("Computed NMISS and OBS_CT", time.perf_counter() - t0)

    # === Chromosome update ===
    t0 = time.perf_counter()
    # Remove 'chr' or 'ch' prefix (case-insensitive)
    munge["chr_update"] = munge["scaffold"].str.replace(r"^chr?|^ch", "", regex=True, case=False)
    log_step("Created chr_update field", time.perf_counter() - t0)

    # === Frequency + region filtering ===
    t0 = time.perf_counter()
    before = len(munge)

    munge["freq.a"] = pd.to_numeric(munge["freq.a"], errors="coerce")
    munge["position"] = pd.to_numeric(munge["position"], errors="coerce")

    condition1 = munge["freq.a"].between(0.01, 0.99)

    condition2 = (
        ((munge["chr_update"].astype(str) == "4") & munge["position"].between(89674099, 89888304))
        | ((munge["chr_update"].astype(str) == "1") & munge["position"].between(155184452, 155294627))
        | ((munge["chr_update"].astype(str) == "12") & munge["position"].between(40174997, 40419285))
    )

    munge = munge[condition1 | condition2]
    log_rows("After frequency (0.01–0.99) + locus-keep filtering", before, len(munge), time.perf_counter() - t0)

    # === Basic cleaning of effect / stderr / pvalue ===
    t0 = time.perf_counter()
    munge["stderr"] = pd.to_numeric(munge["stderr"], errors="coerce")
    munge["pvalue"] = pd.to_numeric(munge["pvalue"], errors="coerce")

    before = len(munge)
    munge = munge[
        munge["effect"].notna()
        & munge["stderr"].notna()
        & munge["pvalue"].notna()
        & munge["freq.a"].notna()
    ]
    log_rows("After dropping rows with NaN effect / stderr / pvalue / freq.a", before, len(munge), time.perf_counter() - t0)

    # === Harmonization with gwaslab ===
    t0 = time.perf_counter()
    logging.info("Initializing harmonization with gwaslab (23andMe-style columns)...")

    mysumstats = gl.Sumstats(
        munge,
        snpid="assay.name",
        chrom="chr_update",
        pos="position",
        p="pvalue",
        build="38",
        nea="alt_allele",
        ea="effect_allele",
        eaf="freq.a",
        beta="effect",
        se="stderr",
        n="OBS_CT",
        other=["NMISS"],
    )

    # --- Check reference paths before calling harmonize ---
    ref_seq_path = gl.get_path("ucsc_genome_hg38")
    ref_infer_path = gl.get_path("1kg_afr_hg38")

    def _check_ref(name, path):
        if not isinstance(path, str) or not os.path.exists(path):
            logging.error(
                f"Could not find gwaslab reference '{name}'.\n"
                f"  gl.get_path('{name}') returned: {path!r}\n"
                f"Please run in Python (once in this environment):\n"
                f"  >>> import gwaslab as gl\n"
                f"  >>> gl.download_ref('{name}')\n"
            )
            sys.exit(1)

    _check_ref("ucsc_genome_hg38", ref_seq_path)
    _check_ref("1kg_afr_hg38", ref_infer_path)

    logging.info("Running harmonization (can take a while)...")
    mysumstats.harmonize(
        basic_check=True,
        n_cores=1,
        ref_seq=ref_seq_path,
        ref_infer=ref_infer_path,
        ref_alt_freq="AF",
    )
    log_step("Harmonization complete", time.perf_counter() - t0)

    # === Extract harmonized data ===
    t0 = time.perf_counter()
    sumdf = mysumstats.data
    log_step(f"Extracted harmonized dataframe with {len(sumdf):,} rows", time.perf_counter() - t0)

    # === NEW_ID generation + dedup ===
    t0 = time.perf_counter()
    logging.info("Generating NEW_IDs")
    tqdm.pandas(desc="Creating NEW_IDs")
    sumdf["NEW_ID"] = (
        "chr"
        + sumdf["CHR"].astype(str).progress_map(str)
        + ":"
        + sumdf["POS"].astype(str)
        + ":"
        + sumdf["NEA"].astype(str)
        + ":"
        + sumdf["EA"].astype(str)
    )
    log_step("NEW_ID generation done", time.perf_counter() - t0)

    t0 = time.perf_counter()
    before = len(sumdf)
    sumdf_nodup = sumdf.drop_duplicates(subset=["NEW_ID"])
    log_rows("After dropping duplicate NEW_IDs", before, len(sumdf_nodup), time.perf_counter() - t0)

    # === Final output (including EAF) ===
    t0 = time.perf_counter()
    cols_to_keep = ["NEW_ID", "CHR", "POS", "EA", "NEA", "EAF", "BETA", "SE", "P", "NMISS"]
    missing_cols = [c for c in cols_to_keep if c not in sumdf_nodup.columns]
    if missing_cols:
        logging.warning(
            f"Missing expected columns in harmonized data: {missing_cols}. "
            f"Proceeding without them where necessary."
        )
        cols_to_keep = [c for c in cols_to_keep if c in sumdf_nodup.columns]

    sumdf_nodup = sumdf_nodup[cols_to_keep]

    rename_map = {
        "NEW_ID": "SNP",
        "CHR": "CHR",
        "POS": "BP",
        "EA": "A1",
        "NEA": "A2",
        "EAF": "EAF",
        "BETA": "BETA",
        "SE": "SE",
        "P": "P",
        "NMISS": "NMISS",
    }
    sumdf_nodup = sumdf_nodup.rename(columns=rename_map)

    logging.info(f"Saving output to {args.output}...")
    sumdf_nodup.to_csv(args.output, index=False, sep="\t", header=True)
    log_step(f"Done | total rows: {len(sumdf_nodup):,}", time.perf_counter() - t0)

    total_time = time.perf_counter() - global_start
    logging.info(f"=== Pipeline finished in {total_time/60:.2f} minutes ({total_time:.2f} seconds) ===")


if __name__ == "__main__":
    main()


In [ ]:
%%bash

cat << EOF > ${WORK_DIR}/scripts/munge_sum_stats_23andMe.swarm
/data/conda/envs/gwaslab_310/bin/python ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_23andMe.py \\
  --input ${DATA_DIR}/preprocessing/filtered_sumstats_23andme_AFRICAN_PD.txt.gz \\
  --output ${DATA_DIR}/23andMe/23andMe_AAC.wAlleles.FOR_PLINK.txt \\
  --log ${WORK_DIR}/scripts/munge_sum_stats_23andMe.wAlleles.FOR_PLINK.log
EOF

In [18]:
%%bash

cd ${WORK_DIR}/scripts/

swarm --file ${WORK_DIR}/scripts/munge_sum_stats_23andMe.swarm \
-g 100 --time 6:00:00 \
--sbatch "--mail-type=BEGIN,FAIL,TIME_LIMIT_80,END"

4830512


In [ ]:
! wc -l {DATA_DIR}/23andMe/23andMe_AAC.wAlleles.FOR_PLINK.txt
! head {DATA_DIR}/23andMe/23andMe_AAC.wAlleles.FOR_PLINK.txt

# MVP

In [38]:
## unfiltered data 
! head {DATA_DIR}/preprocessing/MVP_R4.1000G_AGR.Phe_332.AFR.GIA.dbGaP.txt 

SNP_ID	chrom	pos	ref	alt	ea	af	num_samples	case_af	num_cases	control_af	num_controls	or	ci	pval	r2	q_pval	i2	direction
rs561109771	1	11063	G	T	T	0.9945	121604	0.994	711	0.9945	120893	0.7008	0.2046,2.4	0.5714	0.3437	NA	NA	NA
rs554760071	1	13483	C	G	G	0.9975	121604	0.9972	711	0.9975	120893	0.6037	0.1003,3.635	0.5817	0.368	NA	NA	NA
rs541172944	1	16071	A	G	G	0.9947	121604	0.996	711	0.9947	120893	2.058	0.5769,7.339	0.266	0.3394	NA	NA	NA
rs557560597	1	30998	T	C	C	0.9986	121604	0.9988	711	0.9986	120893	1.277	0.1035,15.75	0.8489	0.3117	NA	NA	NA
rs553367669	1	46716	T	C	C	0.9981	121604	0.9988	711	0.9981	120893	3.387	0.4192,27.37	0.2525	0.3159	NA	NA	NA
rs565824523	1	48327	A	C	C	0.9925	121604	0.9914	711	0.9925	120893	0.7308	0.3165,1.687	0.4626	0.5354	NA	NA	NA
rs528394432	1	48328	T	A	A	0.9925	121604	0.9914	711	0.9925	120893	0.7308	0.3165,1.687	0.4626	0.5354	NA	NA	NA
rs551041711	1	51427	G	T	T	0.999	121604	0.9993	711	0.999	120893	2.069	0.1257,34.06	0.6107	0.3604	NA	NA	NA
rs532124873	1	56813	T	C	C	0.9

In [ ]:
%%writefile ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_MVP.py
#!/usr/bin/env python3.10

import pandas as pd
import numpy as np
from tqdm import tqdm
import gwaslab as gl
import logging
import sys
import time
import os
import argparse
from scipy.stats import norm

"""
Example:

/data/conda/envs/gwaslab_310/bin/python \
  ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_23andMe.py \
  --input  /path/to/MVP_sumstats.txt[.gz] \
  --output /path/to/munged/MVP.stats.wAlleles.FOR_PLINK.txt \
  --log    /path/to/logs/MVP.stats.wAlleles.FOR_PLINK.log

This version is adapted for MVP-style columns, e.g.:

    SNP_ID, chrom, pos, ref, alt, ea, af,
    num_samples, case_af, num_cases, control_af, num_controls,
    or, ci, pval, r2, q_pval, i2, direction

It:
  - builds A2 from (ea, ref, alt)
  - computes BETA from OR
  - computes SE from BETA and pval
  - uses per-variant num_cases / num_controls if present,
    otherwise falls back to constant ncases / ncontrols
  - applies MAF + locus-keep filtering
  - harmonizes with ucsc_genome_hg38 + 1kg_eur_hg38
  - outputs PLINK-ready file with SNP/CHR/BP/A1/A2/EAF/BETA/SE/P/NMISS
"""

# gwaslab config paths
gl.options.set_option("config","/data/gwaslab/data/config.json")
gl.options.set_option("reference","/data/gwaslab/data/reference.json")
gl.options.set_option("formatbook","/data/gwaslab/data/formatbook.json")
gl.options.set_option("data_directory","/data/gwaslab/.gwaslab/")

# ------------------------------------------------------------------
# Logging helpers
# ------------------------------------------------------------------
def setup_logging(logfile):
    """Setup logging that flushes immediately to file + console."""
    os.makedirs(os.path.dirname(logfile), exist_ok=True)
    open(logfile, "a").close()  # touch file

    file_handler = logging.FileHandler(logfile, mode="a")
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

    root = logging.getLogger()
    root.setLevel(logging.INFO)
    root.handlers = []
    root.addHandler(file_handler)
    root.addHandler(console_handler)


def log_rows(step, before, after, elapsed=None):
    removed = before - after
    msg = f"{step}: {before:,} → {after:,} (removed {removed:,})"
    if elapsed is not None:
        msg += f" | time: {elapsed:.2f}s"
    logging.info(msg)


def log_step(step, elapsed=None):
    msg = step
    if elapsed is not None:
        msg += f" | time: {elapsed:.2f}s"
    logging.info(msg)


# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(
        description="Munge and harmonize MVP GWAS summary stats for PLINK input."
    )
    parser.add_argument("--input", required=True, help="Path to input MVP summary stats (tsv, can be gzipped)")
    parser.add_argument("--output", required=True, help="Path to output harmonized file (tsv)")
    parser.add_argument("--log", required=True, help="Path to log file")

    # optional constants if we *don't* have per-variant num_cases/num_controls
    parser.add_argument("--ncases", type=int, default=8704, help="Fallback number of cases (default: 8704)")
    parser.add_argument("--ncontrols", type=int, default=440252, help="Fallback number of controls (default: 440252)")

    args = parser.parse_args()
    setup_logging(args.log)
    global_start = time.perf_counter()

    # === Load data ===
    t0 = time.perf_counter()
    logging.info(f"Reading input file (compression inferred): {args.input}")
    munge = pd.read_csv(
        args.input,
        sep="\t",
        compression="infer",
        dtype={"pos": int},
    )
    log_step(f"Initial rows: {len(munge):,}", time.perf_counter() - t0)

    required_cols = {"SNP_ID", "chrom", "pos", "ref", "alt", "ea", "af", "or", "pval"}
    missing = required_cols - set(munge.columns)
    if missing:
        logging.error(f"Missing required MVP columns: {missing}\nAvailable: {list(munge.columns)}")
        sys.exit(1)

    # === Build A2 from (ea, ref, alt) ===
    t0 = time.perf_counter()
    munge["A2"] = np.where(munge["ea"] == munge["alt"], munge["ref"], munge["alt"])
    log_step("Computed A2 from ea/ref/alt", time.perf_counter() - t0)

    # === Clean OR and compute BETA ===
    t0 = time.perf_counter()
    munge["or"] = pd.to_numeric(munge["or"], errors="coerce")
    before = len(munge)
    munge = munge[munge["or"].notna() & (munge["or"] > 0)]
    log_rows("After dropping OR NaN or <= 0", before, len(munge), time.perf_counter() - t0)

    t0 = time.perf_counter()
    munge["BETA"] = np.log(munge["or"])
    before = len(munge)
    munge = munge[munge["BETA"] != 0]
    log_rows("After dropping BETA == 0", before, len(munge), time.perf_counter() - t0)
    log_step("Computed BETA = log(OR)", None)

    # === Compute SE from BETA and pval ===
    t0 = time.perf_counter()
    munge["pval"] = pd.to_numeric(munge["pval"], errors="coerce")

    before = len(munge)
    munge = munge[(munge["pval"].notna()) & (munge["pval"] > 0) & (munge["pval"] < 1)]
    log_rows("After enforcing 0 < pval < 1", before, len(munge), time.perf_counter() - t0)

    t0 = time.perf_counter()
    z = norm.ppf(munge["pval"] / 2.0)
    # norm.ppf(p/2) is negative; SE uses absolute value
    munge["se"] = np.abs(munge["BETA"] / z)
    log_step("Computed SE from BETA and pval", time.perf_counter() - t0)

    # === N, NMISS, OBS_CT ===
    t0 = time.perf_counter()
    use_per_variant_n = {"num_cases", "num_controls"}.issubset(munge.columns)

    if use_per_variant_n:
        logging.info("Using per-variant num_cases / num_controls from MVP file.")
        munge["num_cases"] = pd.to_numeric(munge["num_cases"], errors="coerce")
        munge["num_controls"] = pd.to_numeric(munge["num_controls"], errors="coerce")

        before = len(munge)
        munge = munge[munge["num_cases"].notna() & munge["num_controls"].notna()]
        log_rows("After dropping rows with NaN num_cases / num_controls", before, len(munge))

        munge["ncases"] = munge["num_cases"].astype(int)
        munge["ncontrols"] = munge["num_controls"].astype(int)
    else:
        logging.info(
            "num_cases / num_controls not found; "
            f"using constant ncases={args.ncases}, ncontrols={args.ncontrols}."
        )
        munge["ncases"] = int(args.ncases)
        munge["ncontrols"] = int(args.ncontrols)

    # Same NMISS formula as your other scripts: 4 / (1/nc + 1/n0)
    munge["NMISS"] = 4.0 / (1.0 / munge["ncases"] + 1.0 / munge["ncontrols"])
    munge["OBS_CT"] = munge["ncases"] + munge["ncontrols"]
    log_step("Computed NMISS and OBS_CT", time.perf_counter() - t0)

    # === Frequency + region filtering ===
    t0 = time.perf_counter()
    before = len(munge)

    munge["af"] = pd.to_numeric(munge["af"], errors="coerce")
    munge["chrom"] = pd.to_numeric(munge["chrom"], errors="coerce")
    munge["pos"] = pd.to_numeric(munge["pos"], errors="coerce")

    condition1 = munge["af"].between(0.01, 0.99)

    condition2 = (
        ((munge["chrom"] == 4) & munge["pos"].between(89674099, 89888304))
        | ((munge["chrom"] == 1) & munge["pos"].between(155184452, 155294627))
        | ((munge["chrom"] == 12) & munge["pos"].between(40174997, 40419285))
    )

    munge = munge[condition1 | condition2]
    log_rows("After frequency (0.01–0.99) + locus-keep filtering", before, len(munge), time.perf_counter() - t0)

    # === Harmonization with gwaslab ===
    t0 = time.perf_counter()
    logging.info("Initializing harmonization with gwaslab (MVP-style columns)...")

    mysumstats = gl.Sumstats(
        munge,
        snpid="SNP_ID",
        chrom="chrom",
        pos="pos",
        p="pval",
        build="38",
        nea="A2",
        ea="ea",
        eaf="af",
        beta="BETA",
        se="se",
        n="OBS_CT",
        other=["NMISS"],
    )

    # Check reference paths before harmonize
    ref_seq_path = gl.get_path("ucsc_genome_hg38")
    ref_infer_path = gl.get_path("1kg_eur_hg38")

    def _check_ref(name, path):
        if not isinstance(path, str) or not os.path.exists(path):
            logging.error(
                f"Could not find gwaslab reference '{name}'.\n"
                f"  gl.get_path('{name}') returned: {path!r}\n"
                f"Please run in Python (once in this environment):\n"
                f"  >>> import gwaslab as gl\n"
                f"  >>> gl.download_ref('{name}')\n"
            )
            sys.exit(1)

    _check_ref("ucsc_genome_hg38", ref_seq_path)
    _check_ref("1kg_afr_hg38", ref_infer_path)

    logging.info("Running harmonization (can take a while)...")
    mysumstats.harmonize(
        basic_check=True,
        n_cores=1,
        ref_seq=ref_seq_path,
        ref_infer=ref_infer_path,
        ref_alt_freq="AF",
    )
    log_step("Harmonization complete", time.perf_counter() - t0)

    # === Extract harmonized data ===
    t0 = time.perf_counter()
    sumdf = mysumstats.data
    log_step(f"Extracted harmonized dataframe with {len(sumdf):,} rows", time.perf_counter() - t0)

    # === NEW_ID generation + dedup ===
    t0 = time.perf_counter()
    logging.info("Generating NEW_IDs")
    tqdm.pandas(desc="Creating NEW_IDs")
    sumdf["NEW_ID"] = (
        "chr"
        + sumdf["CHR"].astype(str).progress_map(str)
        + ":"
        + sumdf["POS"].astype(str)
        + ":"
        + sumdf["NEA"].astype(str)
        + ":"
        + sumdf["EA"].astype(str)
    )
    log_step("NEW_ID generation done", time.perf_counter() - t0)

    t0 = time.perf_counter()
    before = len(sumdf)
    sumdf_nodup = sumdf.drop_duplicates(subset=["NEW_ID"])
    log_rows("After dropping duplicate NEW_IDs", before, len(sumdf_nodup), time.perf_counter() - t0)

    # === Final output (including EAF if present) ===
    t0 = time.perf_counter()
    cols_to_keep = ["NEW_ID", "CHR", "POS", "EA", "NEA", "EAF", "BETA", "SE", "P", "NMISS"]
    missing_cols = [c for c in cols_to_keep if c not in sumdf_nodup.columns]
    if missing_cols:
        logging.warning(
            f"Missing expected columns in harmonized data: {missing_cols}. "
            f"Proceeding without them where necessary."
        )
        cols_to_keep = [c for c in cols_to_keep if c in sumdf_nodup.columns]

    sumdf_nodup = sumdf_nodup[cols_to_keep]

    rename_map = {
        "NEW_ID": "SNP",
        "CHR": "CHR",
        "POS": "BP",
        "EA": "A1",
        "NEA": "A2",
        "EAF": "EAF",
        "BETA": "BETA",
        "SE": "SE",
        "P": "P",
        "NMISS": "NMISS",
    }
    sumdf_nodup = sumdf_nodup.rename(columns=rename_map)

    logging.info(f"Saving output to {args.output}...")
    sumdf_nodup.to_csv(args.output, index=False, sep="\t", header=True)
    log_step(f"Done | total rows: {len(sumdf_nodup):,}", time.perf_counter() - t0)

    total_time = time.perf_counter() - global_start
    logging.info(f"=== MVP pipeline finished in {total_time/60:.2f} minutes ({total_time:.2f} seconds) ===")


if __name__ == "__main__":
    main()


In [ ]:
%%bash

cat << EOF > ${WORK_DIR}/scripts/munge_sum_stats_MVP.swarm
/data/conda/envs/gwaslab_310/bin/python ../2025_2026_AFR_AAC_GWAS/scripts/munge_sum_stats_MVP.py \\
  --input ${DATA_DIR}/preprocessing/MVP_R4.1000G_AGR.Phe_332.AFR.GIA.dbGaP.txt.gz \\
  --output ${DATA_DIR}/MVP/MVP_AAC.wAlleles.FOR_PLINK.txt \\
  --log ${WORK_DIR}/scripts/munge_sum_stats_MVP.wAlleles.FOR_PLINK.log
EOF

In [24]:
%%bash

cd ${WORK_DIR}/scripts/

swarm --file ${WORK_DIR}/scripts/munge_sum_stats_MVP.swarm \
-g 100 --time 6:00:00 \
--sbatch "--mail-type=BEGIN,FAIL,TIME_LIMIT_80,END"

4845352
